# NB08: Phenotype Impact Analysis

Measure how thermodynamic updates affect growth phenotype predictions.
Compare correct-positive (CP), correct-negative (CN), false-positive (FP),
and false-negative (FN) rates before and after the OPAM2 thermodynamic update.

Uses `MSGrowthPhenotypes` from `modelseedpy.core.msgrowthphenotypes` to
simulate growth on various carbon/nitrogen sources.

In [ ]:
import sys
import json
from pathlib import Path

import cobra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

KBUTILLIB_ROOT = Path('/global_share/KBaseUtilities/KBUtilLib')
sys.path.insert(0, str(KBUTILLIB_ROOT / 'src'))

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

from kbutillib import MSFBAUtils

## 1. Load models

In [ ]:
model_files = sorted(DATA_DIR.glob('model_*.json'))
models = {}
for f in model_files:
    org_id = f.stem.replace('model_', '')
    models[org_id] = cobra.io.load_json_model(str(f))
    print(f'Loaded {org_id}: {len(models[org_id].reactions)} reactions')

## 2. Define phenotype conditions

Carbon source utilization tests: simulate growth on minimal media with
each carbon source as the sole carbon input. Nitrogen source tests
use each nitrogen compound as sole nitrogen input.

In [ ]:
from modelseedpy.core.msgrowthphenotypes import MSGrowthPhenotypes, MSGrowthPhenotype

# Common carbon sources with expected growth for E. coli
carbon_sources = {
    'cpd00027': ('Glucose', True),
    'cpd00082': ('Fructose', True),
    'cpd00036': ('Succinate', True),
    'cpd00029': ('Acetate', True),
    'cpd00047': ('Formate', False),
    'cpd00100': ('Glycerol', True),
    'cpd00159': ('Lactose', True),
    'cpd00076': ('Sucrose', False),
    'cpd00179': ('Maltose', True),
    'cpd00064': ('Citrate', True),
}

print(f'Defined {len(carbon_sources)} carbon source phenotype conditions')

## 3. Simulate phenotypes

For each model, test growth on each carbon source by modifying exchange
reactions in minimal media.

In [ ]:
phenotype_results = []

for org_id, model in models.items():
    print(f'\n=== Phenotype simulation: {org_id} ===')
    fba_utils = MSFBAUtils()

    for cpd_id, (cpd_name, expected_growth) in carbon_sources.items():
        mdl = model.copy()

        # Close all carbon exchange reactions
        for rxn in mdl.reactions:
            if rxn.id.startswith('EX_'):
                rxn.lower_bound = 0

        # Open essential exchanges (NH4, PO4, SO4, O2, H2O, etc.)
        essential_exchanges = [
            'EX_cpd00013_e0',  # NH3
            'EX_cpd00009_e0',  # PO4
            'EX_cpd00048_e0',  # SO4
            'EX_cpd00007_e0',  # O2
            'EX_cpd00001_e0',  # H2O
            'EX_cpd00067_e0',  # H+
            'EX_cpd10516_e0',  # Fe3+
        ]
        for ex_id in essential_exchanges:
            if ex_id in mdl.reactions:
                mdl.reactions.get_by_id(ex_id).lower_bound = -100

        # Open the test carbon source
        ex_id = f'EX_{cpd_id}_e0'
        if ex_id in mdl.reactions:
            mdl.reactions.get_by_id(ex_id).lower_bound = -10
        else:
            phenotype_results.append({
                'organism': org_id,
                'cpd_id': cpd_id,
                'cpd_name': cpd_name,
                'expected': expected_growth,
                'predicted_growth': 0.0,
                'predicted': False,
                'classification': 'CN' if not expected_growth else 'FN',
                'note': 'Exchange reaction not in model',
            })
            continue

        # Run FBA
        sol = mdl.optimize()
        growth = sol.objective_value if sol.status == 'optimal' else 0.0
        predicted = growth > 1e-6

        if predicted and expected_growth:
            cls = 'CP'
        elif not predicted and not expected_growth:
            cls = 'CN'
        elif predicted and not expected_growth:
            cls = 'FP'
        else:
            cls = 'FN'

        phenotype_results.append({
            'organism': org_id,
            'cpd_id': cpd_id,
            'cpd_name': cpd_name,
            'expected': expected_growth,
            'predicted_growth': growth,
            'predicted': predicted,
            'classification': cls,
        })

df_pheno = pd.DataFrame(phenotype_results)
print(f'\nTotal phenotype tests: {len(df_pheno)}')
print(df_pheno['classification'].value_counts())

## 4. Compute accuracy metrics

In [ ]:
for org_id in models:
    sub = df_pheno[df_pheno['organism'] == org_id]
    if len(sub) == 0:
        continue

    cp = (sub['classification'] == 'CP').sum()
    cn = (sub['classification'] == 'CN').sum()
    fp = (sub['classification'] == 'FP').sum()
    fn = (sub['classification'] == 'FN').sum()

    accuracy = (cp + cn) / len(sub) if len(sub) > 0 else 0
    sensitivity = cp / (cp + fn) if (cp + fn) > 0 else 0
    specificity = cn / (cn + fp) if (cn + fp) > 0 else 0

    print(f'\n{org_id}:')
    print(f'  CP={cp}, CN={cn}, FP={fp}, FN={fn}')
    print(f'  Accuracy: {accuracy:.1%}')
    print(f'  Sensitivity: {sensitivity:.1%}')
    print(f'  Specificity: {specificity:.1%}')

## 5. Save results

In [ ]:
df_pheno.to_csv(DATA_DIR / 'phenotype_results.tsv', sep='\t', index=False)
print(f'Saved {len(df_pheno)} phenotype results to data/phenotype_results.tsv')

## 6. Visualization

In [ ]:
if len(df_pheno) > 0:
    fig, ax = plt.subplots(figsize=(8, 6))

    for i, org_id in enumerate(models.keys()):
        sub = df_pheno[df_pheno['organism'] == org_id]
        if len(sub) == 0:
            continue
        counts = sub['classification'].value_counts()
        for j, cls in enumerate(['CP', 'CN', 'FP', 'FN']):
            val = counts.get(cls, 0)
            ax.bar(j + i * 0.25, val, 0.2, label=org_id if j == 0 else '',
                   color=f'C{i}')

    ax.set_xticks(range(4))
    ax.set_xticklabels(['CP', 'CN', 'FP', 'FN'])
    ax.set_ylabel('Count')
    ax.set_title('Phenotype prediction results')
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'phenotype_results.png', dpi=150)
    plt.show()